<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><b><font size=10>Data Science and Business Analytics</font></b></center>
<center><b><font size=6>Unsupervised Learning</font></b></center>

<center><img src="https://cdn.pixabay.com/photo/2015/11/08/16/13/currency-1033803_1280.jpg" width="720"></center>

<center><font size=6>Economic Freedom Analysis</font></center>

# **Problem Statement**

## Context

Created in 1995 by the Heritage Foundation, The Index of Economic Freedom is a ranking created to measure the economic freedom in the countries of the world. Now, in its 25th edition, The Economic Freedom Index is poised to help readers track over two decades of the advancement in economic freedom, prosperity, and opportunity and promote these ideas in their homes, schools, and communities. The Index covers 12 freedoms, from property rights to financial freedom, in 186 countries.

## Objective

To analyze the data, use clustering algorithms to identify different groups of countries based on economic freedom, and list the insights from the analysis.

## Data Description

The data comprises factors indicating economic freedom. The list of variables in the data is given below. All these features are self-explanatory and more details can be found in the data source listed below.

- CountryID
- Country Name
- WEBNAME
- Region
- World Rank
- Region Rank
- 2019 Score
- Property Rights
- Judical Effectiveness
- Government Integrity
- Tax Burden
- Gov't Spending
- Fiscal Health
- Business Freedom
- Labor Freedom
- Monetary Freedom
- Trade Freedom
- Investment Freedom
- Financial Freedom
- Tariff Rate (%)
- Income Tax Rate (%)
- Corporate Tax Rate (%)
- Tax Burden % of GDP
- Gov't Expenditure % of GDP
- Country
- Population (Millions)
- GDP (Billions, PPP)
- GDP Growth Rate (%)
- 5 Year GDP Growth Rate (%)
- GDP per Capita (PPP)
- Unemployment (%)
- Inflation (%)
- FDI Inflow (Millions)
- Public Debt (% of GDP)

**Data Source**

This dataset belongs to The Heritage Foundation and is freely available to download on their [website](https://www.heritage.org/index).

# **Importing necessary libraries**

In [ ]:
# Libraries to help with reading and manipulating data
import pandas as pd
import numpy as np

# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)

# to scale the data using z-score
from sklearn.preprocessing import StandardScaler

# to compute distances
from scipy.spatial.distance import cdist, pdist

# to perform k-means clustering and compute silhouette scores
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# to visualize the elbow curve and silhouette scores
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

# to perform hierarchical clustering, compute cophenetic correlation, and create dendrograms
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet

# to perform PCA
from sklearn.decomposition import PCA

# to suppress warnings
import warnings

warnings.filterwarnings("ignore")

# **Loading the dataset**

In [ ]:
# loading the dataset
data = pd.read_csv("/content/economic_freedom_index2019_data.csv", encoding="latin-1")

# **Overview of the Dataset**

The initial steps to get an overview of any dataset is to:
- observe the first few rows of the dataset, to check whether the dataset has been loaded properly or not
- get information about the number of rows and columns in the dataset
- find out the data types of the columns to ensure that data is stored in the preferred format and the value of each property is as expected.
- check the statistical summary of the dataset to get an overview of the numerical columns of the data

## Displaying first 5 and last 5 rows of the dataset

In [ ]:
# viewing the first 5 rows of the data
data.head()

In [ ]:
data.tail()

## Checking the shape of the dataset

In [ ]:
data.shape

* The dataset has 186 rows and 34 columns

## Creating a copy of original data

In [ ]:
# copying the data to another variable to avoid any changes to original data
df = data.copy()

In [ ]:
# we will drop the CountryID column add it adds no value to the analysis
df.drop("CountryID", axis=1, inplace=True)

## Checking the data types of the columns for the dataset

In [ ]:
# checking datatypes and number of non-null values for each column
df.info()

**Observations**

- There are quite a few columns with null entries.
- *Population (Millions)*, *GDP (Billions, PPP)*, *GDP per Capita (PPP)*, *Unemployment (%)*, and *FDI Inflow (Millions)* are of *object* type in the data but are actually numeric in nature.

## Checking the missing values

In [ ]:
# checking for missing values
df.isnull().sum()

- As seen previously, many columns have null values.

## Checking for duplicates values

In [ ]:
# checking for duplicate values
df.duplicated().sum()

- There are no duplicate values in the data.

# **Data Preprocessing**

## Processing columns



- Let's process the `Population (Millions)`, `GDP (Billions, PPP)`, `GDP per Capita (PPP)`, `Unemployment (%)`, and `FDI Inflow (Millions)` columns to extract numerical values from them.

### **1. `Population (Millions)`**



- Let's first remove the leading and trailing spaces from the column values.

In [ ]:
col = "Population (Millions)"

df[col] = df[col].str.strip()

- Let's check if there are any string values left in the columns.

In [ ]:
for i in range(df.shape[0]):
    item = df.loc[i, col]

    # checking if the value is NaN (NaN values are not equal to themselves)
    if item == item:
        # replacing the decimal point by a blank string
        # and checking if the remaining characters are digits
        if item.replace(".", "").isdigit() == False:
            print(i, "\t", item)

- We will replace this string value with an appropriate number.

In [ ]:
# replacing the value
df.loc[99, col] = 38000

# changing column datatype
df[col] = df[col].astype(float)

### **2. `GDP (Billions, PPP)` and `GDP per Capita (PPP)`**



- We saw that the *GDP (Billions, PPP)* and *GDP per Capita (PPP)* column values had a \\$ symbol at the start and commas in between.
- Let's replace the \\$ symbols and commas by a blank string.
- We will also remove the leading and trailing spaces.

In [ ]:
df["GDP (Billions, PPP)"] = (
    df["GDP (Billions, PPP)"]
    .str.replace(",", "")
    .str.replace("$", "", regex=False)
    .str.strip()
)
df["GDP per Capita (PPP)"] = (
    df["GDP per Capita (PPP)"]
    .str.replace(",", "")
    .str.replace("$", "", regex=False)
    .str.strip()
)
df.head()

- Let's check if there are any string values left in the columns.

In [ ]:
col = "GDP (Billions, PPP)"

for i in range(df.shape[0]):
    item = df.loc[i, col]

    # checking if the value is NaN (NaN values are not equal to themselves)
    if item == item:
        # replacing the decimal point by a blank string
        # and checking if the remaining characters are digits
        if item.replace(".", "").isdigit() == False:
            print(i, "\t", item)

- We will replace these string values with appropriate numbers.

In [ ]:
# replacing the value
df.loc[88, col] = 40.0
df.loc[99, col] = 6.1

# changing column datatype
df[col] = df[col].astype(float)

In [ ]:
col = "GDP per Capita (PPP)"

for i in range(df.shape[0]):
    item = df.loc[i, col]

    # checking if the value is NaN (NaN values are not equal to themselves)
    if item == item:
        # replacing the decimal point by a blank string
        # and checking if the remaining characters are digits
        if item.replace(".", "").isdigit() == False:
            print(i, "\t", item)

- We will replace these string values with appropriate numbers.

In [ ]:
# replacing the value
df.loc[88, col] = 1700
df.loc[99, col] = 139100

# changing column datatype
df[col] = df[col].astype(float)

### **3. `Unemployment (%)`**



- Let's first remove the leading and trailing spaces from the column values.

In [ ]:
col = "Unemployment (%)"

df[col] = df[col].str.strip()

- Let's check if there are any string values left in the columns.

In [ ]:
for i in range(df.shape[0]):
    item = df.loc[i, col]

    # checking if the value is NaN (NaN values are not equal to themselves)
    if item == item:
        # replacing the decimal point by a blank string
        # and checking if the remaining characters are digits
        if item.replace(".", "").isdigit() == False:
            print(i, "\t", item)

- We will replace this string value with an appropriate number.

In [ ]:
# replacing the value
df.loc[99, col] = 2.1

# changing column datatype
df[col] = df[col].astype(float)

### **4. `FDI Inflow (Millions)`**



- We saw that the *FDI Inflow (Millions)* column values had commas in between.
- Let's replace the commas by a blank string.
- We will also remove the leading and trailing spaces.

In [ ]:
col = "FDI Inflow (Millions)"

df[col] = df[col].str.replace(",", "").str.strip()
df.head()

- Let's check if there are any string values left in the columns.

In [ ]:
for i in range(df.shape[0]):
    item = df.loc[i, col]

    # checking if the value is NaN (NaN values are not equal to themselves)
    if item == item:
        # replacing the decimal point by a blank string
        # and checking if the remaining characters are digits
        if item.replace(".", "").isdigit() == False:
            print(i, "\t", item)

- None of the values look problematic and we can convert the column to numeric.

In [ ]:
# changing column datatype
df[col] = df[col].astype(float)

In [ ]:
df.info()

## Statistical summary of the dataset

In [ ]:
# let's look at the statistical summary of the data
df.describe(include="all").T

**Observations**

- The 2019 score varies from 5.9 to 90.2.
- There are 5 regions.
- The income tax rate varies from 0-60%.
- There are a few missing values in the data.

# **Exploratory Data Analysis**

## Functions for EDA

In [ ]:
# function to plot a boxplot and a histogram along the same scale.


def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

## Univariate analysis

`2019 Score`

In [ ]:
histogram_boxplot(df, "2019 Score")

- 2019 sore looks slightly normally distributed with a few lower outliers, and the mean and median scores are close to 60.

`Income Tax Rate (%)`

In [ ]:
histogram_boxplot(df, "Income Tax Rate (%)")

- Income tax rate varies between 0% to 60%, with a median of 30%.

`Population (Millions)`

In [ ]:
histogram_boxplot(df, "Population (Millions)", bins=50)

- Population is heavily right-skewed.

`Region`

In [ ]:
labeled_barplot(df, "Region")

- The region of Sub-Saharan Africa has the highest number of countries in the data, while Middle East and North America has the lowest.

In [ ]:
cols_list = df.select_dtypes(include=np.number).columns.to_list()
cols_list.remove("World Rank")
cols_list.remove("Region Rank")

fig, axes = plt.subplots(9, 3, figsize=(25, 25))
counter = 0

for ii in range(9):
    sns.ecdfplot(ax=axes[ii][0], x=df[cols_list[counter]])
    counter = counter + 1

    sns.ecdfplot(ax=axes[ii][1], x=df[cols_list[counter]])
    counter = counter + 1

    sns.ecdfplot(ax=axes[ii][2], x=df[cols_list[counter]])
    counter = counter + 1

fig.tight_layout(pad=2.0)

**Observations**

- 50% of countries have a 2019 score of less than 60.
- 80% of countries have properties right score less than 70.
- 80% of countries have a Judicial Effectiveness score of less than 60.
- 80% of countries have Govt integrity less than 50.
- 20% of countries have a tax burden of less than 70.
- 40% of countries have govt spending less than 63.
- 40% of countries have fiscal health less than 70.
- 40% of countries have Business freedom less than 60.
- 60% of countries have labor freedom less than 63.
- 20% of countries have monetary freedom less than 70.
- 60% of countries have trade freedom less than 80.
- 80% of countries have a Tariff Rate of less than 10%.
- 80% of countries have an income tax rate of less than 40%.
- 80% of countries have a corporate tax rate of less than 30%.
- 80% of countries have a Tax burden of less than 33% of their GDP.

## Bivariate Analysis

**Let's check for correlations.**

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(
    df[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".1f", cmap="Spectral"
)
plt.show()

In [ ]:
sns.pairplot(data=df[cols_list], diag_kind="kde")
plt.show()

**Observations**

- *Tariff Rate (%)* and *Trade Freedom* are perfectly negatively correlated.
- *2019 Score* is highly positively correlated with many variables.

We will drop the *Tariff Rate (%)* column from the data.

In [ ]:
df.drop("Tariff Rate (%)", axis=1, inplace=True)

# **Data Preprocessing (Continued)**

## Missing Value Treatment

In [ ]:
df.isna().sum()

In [ ]:
# let's impute missing values with column median
df = df.fillna(df.median(numeric_only = True))

In [ ]:
# let's see if missing values has been imputed
df.isna().sum()

* Missing values are imputed.

## Scaling

- Let's scale the data before we proceed with clustering.

In [ ]:
scaler = StandardScaler()
to_exclude = [
    "CountryID",
    "Country Name",
    "WEBNAME",
    "Region",
    "World Rank",
    "Region Rank",
    "Country",
]

# columns to be used for clustering are
col_4_clustering = [c for c in df.columns if c not in to_exclude]
print(col_4_clustering)

In [ ]:
df_scaled = df[col_4_clustering].copy()
df_scaled.iloc[:, :] = scaler.fit_transform(df_scaled.iloc[:, :])
df_scaled.head()

In [ ]:
# creating dataframe copies for k-means and hierarchical clustering
km_df = df.copy()
hc_df = df.copy()

# **Model Building**

## K-means Clustering

### Checking Elbow Plot

In [ ]:
clusters = range(2, 11)
meanDistortions = []

for k in clusters:
    model = KMeans(n_clusters=k, random_state=1)
    model.fit(df_scaled)
    prediction = model.predict(df_scaled)
    distortion = (
        sum(np.min(cdist(df_scaled, model.cluster_centers_, "euclidean"), axis=1))
        / df_scaled.shape[0]
    )

    meanDistortions.append(distortion)

    print("Number of Clusters:", k, "\tAverage Distortion:", distortion)

plt.plot(clusters, meanDistortions, "bx-")
plt.xlabel("k")
plt.ylabel("Average Distortion")
plt.title("Selecting k with the Elbow Method", fontsize=20)
plt.show()

**Let's do further analysis to determine the optimal value of k**

In [ ]:
model = KMeans(random_state=1)
visualizer = KElbowVisualizer(model, k=(2, 11), timings=True)
visualizer.fit(df_scaled)  # fit the data to the visualizer
visualizer.show()  # finalize and render figure

### Checking Silhouette Scores

**Let's check the silhouette scores.**

In [ ]:
sil_score = []
cluster_list = range(2, 11)
for n_clusters in cluster_list:
    clusterer = KMeans(n_clusters=n_clusters, random_state=1)
    preds = clusterer.fit_predict((df_scaled))
    score = silhouette_score(df_scaled, preds)
    sil_score.append(score)
    print("For n_clusters = {}, the silhouette score is {})".format(n_clusters, score))

plt.plot(cluster_list, sil_score)
plt.show()

In [ ]:
model = KMeans(random_state=1)
visualizer = KElbowVisualizer(model, k=(2, 11), metric="silhouette", timings=True)
visualizer.fit(df_scaled)  # fit the data to the visualizer
visualizer.show()  # finalize and render figure

**From the silhouette scores, it seems that 4 is a good value for k.**

**Silhouette Plot**

In [ ]:
# Finding optimal no. of clusters with silhouette coefficients
visualizer = SilhouetteVisualizer(KMeans(2, random_state=1))
visualizer.fit(df_scaled)
visualizer.show()

In [ ]:
# Finding optimal no. of clusters with silhouette coefficients
visualizer = SilhouetteVisualizer(KMeans(3, random_state=1))
visualizer.fit(df_scaled)
visualizer.show()

In [ ]:
# Finding optimal no. of clusters with silhouette coefficients
visualizer = SilhouetteVisualizer(KMeans(4, random_state=1))
visualizer.fit(df_scaled)
visualizer.show()

In [ ]:
# Finding optimal no. of clusters with silhouette coefficients
visualizer = SilhouetteVisualizer(KMeans(5, random_state=1))
visualizer.fit(df_scaled)
visualizer.show()

In [ ]:
# Finding optimal no. of clusters with silhouette coefficients
visualizer = SilhouetteVisualizer(KMeans(6, random_state=1))
visualizer.fit(df_scaled)
visualizer.show()

In [ ]:
# Finding optimal no. of clusters with silhouette coefficients
visualizer = SilhouetteVisualizer(KMeans(7, random_state=1))
visualizer.fit(df_scaled)
visualizer.show()

**We will proceed with k=4**

### Creating Final Model

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=1)
kmeans.fit(df_scaled)

In [ ]:
# adding kmeans cluster labels to the original dataframe
km_df["KM_segments"] = kmeans.labels_

## Hierarchical Clustering

### Computing Cophenetic Correlation

In [ ]:
# list of distance metrics
distance_metrics = ["euclidean", "chebyshev", "mahalanobis", "cityblock"]

# list of linkage methods
linkage_methods = ["single", "complete", "average", "weighted"]

high_cophenet_corr = 0
high_dm_lm = [0, 0]

for dm in distance_metrics:
    for lm in linkage_methods:
        Z = linkage(df_scaled, metric=dm, method=lm)
        c, coph_dists = cophenet(Z, pdist(df_scaled))
        print(
            "Cophenetic correlation for {} distance and {} linkage is {}.".format(
                dm.capitalize(), lm, c
            )
        )
        if high_cophenet_corr < c:
            high_cophenet_corr = c
            high_dm_lm[0] = dm
            high_dm_lm[1] = lm

In [ ]:
# printing the combination of distance metric and linkage method with the highest cophenetic correlation
print(
    "Highest cophenetic correlation is {}, which is obtained with {} distance and {} linkage.".format(
        high_cophenet_corr, high_dm_lm[0].capitalize(), high_dm_lm[1]
    )
)

### Checking Dendrograms

**We see that the cophenetic correlation is maximum with Euclidean distance and average linkage.**


**Let's view the dendrograms for the different linkage methods.**

In [ ]:
# list of linkage methods
linkage_methods = ["single", "complete", "average", "centroid", "ward", "weighted"]

# lists to save results of cophenetic correlation calculation
compare_cols = ["Linkage", "Cophenetic Coefficient"]
compare = []

# to create a subplot image
fig, axs = plt.subplots(len(linkage_methods), 1, figsize=(15, 30))

# We will enumerate through the list of linkage methods above
# For each linkage method, we will plot the dendrogram and calculate the cophenetic correlation
for i, method in enumerate(linkage_methods):
    Z = linkage(df_scaled, metric="euclidean", method=method)

    dendrogram(Z, ax=axs[i])
    axs[i].set_title(f"Dendrogram ({method.capitalize()} Linkage)")

    coph_corr, coph_dist = cophenet(Z, pdist(df_scaled))
    axs[i].annotate(
        f"Cophenetic\nCorrelation\n{coph_corr:0.2f}",
        (0.80, 0.80),
        xycoords="axes fraction",
    )

    compare.append([method, coph_corr])

**Observations**

- Looking the the above dendrograms, the Ward linkage seems to result in the best separation between clusters, even though its cophenetic correlation is lower than the other linkages.
- 4 looks to be a good choice for no. of clusters.

### Creating Final Model

In [ ]:
hc = AgglomerativeClustering(n_clusters=4, affinity="euclidean", linkage="ward")
hc_labels = hc.fit_predict(df_scaled)

In [ ]:
hc_df["HC_segments"] = hc_labels

# **Cluster Profiling and Comparison**

## Cluster Profiling: K-means Clustering

In [ ]:
pd.crosstab(km_df.KM_segments, km_df.Region)

**Observations**

- Majority of the European countries are in Cluster 1 and Cluster 2.
- Majority of Asia-Pacific, American, Middle East and North Africa countries are in Cluster 0 and Cluster 1.
- Almost all Sub-Saharan African countries are in Cluster 0.
- There is only 1 country (American) in cluster 3.

In [ ]:
clusters = km_df.KM_segments.unique().tolist()
for cl in clusters:
    print(
        "The",
        km_df[km_df["KM_segments"] == cl]["Country Name"].nunique(),
        "countries in cluster",
        cl,
        "are:",
    )
    print(km_df[km_df["KM_segments"] == cl]["Country Name"].unique())
    print("-" * 100, "\n")

In [ ]:
km_df_profiling = km_df.groupby("KM_segments")[col_4_clustering].mean()
km_df_profiling["count_in_each_segment"] = (
    km_df.groupby("KM_segments")["2019 Score"].count().values
)

# displaying the group-wise means of variables
km_df_profiling.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
# let's plot the attributes of only the big clusters
c1 = [0, 1, 2]
km_df2 = km_df[km_df.KM_segments.isin(c1)]

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(25, 35))
counter = 0

for ii in range(7):
    for jj in range(4):
        if counter < 26:
            sns.boxplot(
                ax=axes[ii][jj],
                y=km_df2[col_4_clustering[counter]],
                x=km_df2["KM_segments"],
            )
            counter = counter + 1

fig.tight_layout(pad=3.0)

### Insights

#### Cluster 0

  - Countries in this cluster have a lower 2019 Score compared to the other clusters, with a mean value of 52.71.

  - Property Rights, Judicial Effectiveness, and Government Integrity scores are also lower in this cluster compared to the other clusters.

  - Tax Burden is slightly lower in this cluster compared to Cluster 1 and Cluster 2.

  - Government Spending is lower in Cluster 0 compared to the other clusters.

  - Fiscal Health is also lower in Cluster 0 compared to the other clusters.

  - Business Freedom and Labor Freedom scores are moderate in this cluster.

  - Monetary Freedom, Trade Freedom, and Investment Freedom scores are also moderate in Cluster 0.

  - Financial Freedom score is slightly lower in Cluster 0 compared to Cluster 1 and Cluster 2.

  - Income Tax Rate (%) and Corporate Tax Rate (%) are moderate in this cluster.

  - Tax Burden (% of GDP) is slightly lower in Cluster 0 compared to the other clusters.

  - Government Expenditure (% of GDP) is lower in Cluster 0 compared to the other clusters.

  - The Population in this cluster is moderate, with a mean value of 62.27 million.

  - The GDP (Billions, PPP) is also moderate in Cluster 0.

  - GDP Growth Rate (%) is moderate in Cluster 0.

  - 5 Year GDP Growth Rate (%) is also moderate in Cluster 0.

  - GDP per Capita (PPP) is moderate in Cluster 0.

  - The Unemployment Rate (%) is higher in Cluster 0 compared to the other clusters.

  - The Inflation Rate (%) is moderate in Cluster 0.

  - FDI Inflow (Millions) is lower in Cluster 0 compared to the other clusters.

  - Public Debt (% of GDP) is moderate in Cluster 0.

#### Cluster 1

- Countries in this cluster have a higher 2019 Score compared to the other clusters, with a mean value of 70.63.

- Property Rights, Judicial Effectiveness, and Government Integrity scores are also higher in this cluster compared to the other clusters.

- Tax Burden is slightly higher in this cluster compared to Cluster 0 and Cluster 2.

- Government Spending is higher in Cluster 1 compared to the other clusters.

- Fiscal Health is also higher in Cluster 1 compared to the other clusters.

- Business Freedom and Labor Freedom scores are moderate in this cluster.

- Monetary Freedom, Trade Freedom, and Investment Freedom scores are also moderate in Cluster 1.

- Financial Freedom score is slightly higher in Cluster 1 compared to Cluster 0 and Cluster 2.

- Income Tax Rate (%) and Corporate Tax Rate (%) are moderate in this cluster.

- Tax Burden (% of GDP) is slightly higher in Cluster 1 compared to the other clusters.

- Government Expenditure (% of GDP) is higher in Cluster 1 compared to the other clusters.

- The Population in this cluster is high, with a mean value of 519.39 million.

- The GDP (Billions, PPP) is also high in Cluster 1.

- GDP Growth Rate (%) is moderate in Cluster 1.

- 5 Year GDP Growth Rate (%) is also moderate in Cluster 1.

- GDP per Capita (PPP) is moderate in Cluster 1.

-  The Unemployment Rate (%) is moderate in Cluster 1.

- The Inflation Rate (%) is moderate in Cluster 1.

- FDI Inflow (Millions) is high in Cluster 1 compared to the other clusters.

- Public Debt (% of GDP) is moderate in Cluster 1.

#### Cluster 2

- Countries in this cluster have a moderate 2019 Score compared to the other clusters, with a mean value of 54.93.

- Property Rights, Judicial Effectiveness, and Government Integrity scores are also moderate in this cluster.

- Tax Burden is moderate in this cluster.

- Government Spending is moderate in Cluster 2 compared to the other clusters.

- Fiscal Health is also moderate in Cluster 2 compared to the other clusters.

- Business Freedom and Labor Freedom scores are moderate in this cluster.

- Monetary Freedom, Trade Freedom, and Investment Freedom scores are also moderate in Cluster 2.

- Financial Freedom score is moderate in Cluster 2.

- Income Tax Rate (%) and Corporate Tax Rate (%) are moderate in this cluster.

- Tax Burden (% of GDP) is moderate in Cluster 2.

- Government Expenditure (% of GDP) is moderate in Cluster 2.

- The Population in this cluster is moderate, with a mean value of 253.43 million.

- The GDP (Billions, PPP) is also moderate in Cluster 2.

- GDP Growth Rate (%) is moderate in Cluster 2.

- 5 Year GDP Growth Rate (%) is also moderate in Cluster 2.

- GDP per Capita (PPP) is moderate in Cluster 2.

- The Unemployment Rate (%) is moderate in Cluster 2.

- The Inflation Rate (%) is moderate in Cluster 2.

- FDI Inflow (Millions) is moderate in Cluster 2.

- Public Debt (% of GDP) is moderate in Cluster 2.

## Cluster Profiling: Hierarchical Clustering

In [ ]:
pd.crosstab(hc_df.HC_segments, hc_df.Region)

**Observations**

- Majority of the European countries are in Cluster 0 and Cluster 2.
- Majority of Asia-Pacific, American, Middle East and North Africa countries are in Cluster 2 and Cluster 3.
- Almost all Sub-Saharan African countries are in Cluster 3.
- There are 3 countries in Cluster 1.

In [ ]:
clusters = hc_df.HC_segments.unique().tolist()
for cl in clusters:
    print(
        "The",
        hc_df[hc_df["HC_segments"] == cl]["Country Name"].nunique(),
        "countries in cluster",
        cl,
        "are:",
    )
    print(hc_df[hc_df["HC_segments"] == cl]["Country Name"].unique())
    print("-" * 100, "\n")

In [ ]:
hc_df_profiling = hc_df.groupby("HC_segments")[col_4_clustering].mean()
hc_df_profiling["count_in_each_segment"] = (
    hc_df.groupby("HC_segments")["2019 Score"].count().values
)

# displaying the group-wise means of variables
hc_df_profiling.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
# let's plot the attributes of only the big clusters
c1 = [0, 2, 3]
hc_df2 = hc_df[hc_df.HC_segments.isin(c1)]

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(25, 35))
counter = 0

for ii in range(7):
    for jj in range(4):
        if counter < 26:
            sns.boxplot(
                ax=axes[ii][jj],
                y=hc_df2[col_4_clustering[counter]],
                x=hc_df2["HC_segments"],
            )
            counter = counter + 1

fig.tight_layout(pad=2.0)

### Insights

#### Cluster 0

  - There are 34 countries in this cluster.
  - Countries in this cluster have a high 2019 score with a mean value of ~73.4.
  - Countries in this cluster have high business and trade freedom scores.
  - The mean population across countries in this cluster is moderate.
  - Countries in this cluster have a high GDP but low GDP growth rate.
  - The unemployment rate of the countries in this cluster has moderate variance with a comparatively moderate mean.

#### Cluster 2

  - There are 54 countries in this cluster.
  - Countries in this cluster have a moderate 2019 score with a mean value of ~66.
  - Countries in this cluster have moderate business and trade freedom scores.
  - The mean population across countries in this cluster is very high.
  - Countries in this cluster have a moderate GDP and comparatively moderate GDP growth rate.
  - The unemployment rate of the countries in this cluster has a high variance with a comparatively higher mean.

#### Cluster 3

  - There are 95 countries in this cluster.
  - Countries in this cluster have a moderate 2019 score with a mean value of ~54.
  - Countries in this cluster have moderate business and trade freedom scores.
  - The mean population across countries in this cluster is moderate.
  - Countries in this cluster have a moderate GDP and GDP growth rate.
  - The unemployment rate of the countries in this cluster has little variance with a comparatively moderate mean.

## K-means vs Hierarchical Clustering

In [ ]:
km_df_profiling.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
clusters = km_df.KM_segments.unique().tolist()
for cl in clusters:
    print(
        "The",
        km_df[km_df["KM_segments"] == cl]["Country Name"].nunique(),
        "countries in cluster",
        cl,
        "are:",
    )
    print(km_df[km_df["KM_segments"] == cl]["Country Name"].unique())
    print("-" * 100, "\n")

In [ ]:
hc_df_profiling.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
clusters = hc_df.HC_segments.unique().tolist()
for cl in clusters:
    print(
        "The",
        hc_df[hc_df["HC_segments"] == cl]["Country Name"].nunique(),
        "countries in cluster",
        cl,
        "are:",
    )
    print(hc_df[hc_df["HC_segments"] == cl]["Country Name"].unique())
    print("-" * 100, "\n")

**Observations**

- Looks like the K-Means and Hierarchical clusters are similar, except that the labels are swapped between clusters 1 and 2 and some of the countries have changed clusters.
- Let's swap the labels for K-Means for better analysis and comparison.

In [ ]:
km_df["KM_segments"] = km_df["KM_segments"].replace({0: 2, 1: 0, 2: 1})

km_df_profiling = km_df.groupby("KM_segments")[col_4_clustering].mean()
km_df_profiling["count_in_each_segment"] = (
    km_df.groupby("KM_segments")["2019 Score"].count().values
)

In [ ]:
km_df_profiling.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
clusters = km_df.KM_segments.unique().tolist()
for cl in clusters:
    print(
        "The",
        km_df[km_df["KM_segments"] == cl]["Country Name"].nunique(),
        "countries in cluster",
        cl,
        "are:",
    )
    print(km_df[km_df["KM_segments"] == cl]["Country Name"].unique())
    print("-" * 100, "\n")

**Observations**

- The country segments based on economic freedom matches to a large extent for both K-means and Hierarchical Clustering techniques.
- The two clusters with 3 or less countries have one same country - Venezuela for both K-means and Hierarchical clustering.
- Although some of the countries have been swapped between K-means and Hierarchical clusters, we can deduce that overall for both the methodologies, the characteristics of the clusters are similar.
- A few other countries have swapped clusters between K-means and Hierarchical clustering.

# **Add-on: PCA for Visualization**

- Let's use PCA to reduce the data to two dimensions and visualize it to see how well-separated the clusters are.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_reduced_pca = pca.fit_transform(df_scaled)

# storing results in a dataframe
reduced_data_df_pca = pd.DataFrame(
    data=X_reduced_pca, columns=["Component 1", "Component 2"]
)

# checking the amount of variance explained
print(
    f"The first two principal components explain {np.round(100*pca.explained_variance_ratio_.sum(), 2)}% of the variance in the data."
)

In [ ]:
sns.scatterplot(data=reduced_data_df_pca, x="Component 1", y="Component 2")

In [ ]:
sns.scatterplot(
    data=reduced_data_df_pca,
    x="Component 1",
    y="Component 2",
    hue=hc_df["HC_segments"],
    palette="rainbow",
)

- Other than Cluster 1, which has only 3 countries, the clusters are pretty well separated.

<font size=6 color='blue'>Power Ahead</font>
___